<a href="https://colab.research.google.com/github/khushikumari-0811/AI-lab-5th-sem/blob/main/8_Puzzle_By_A__Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import heapq

MOVES = {
    'U': -3,
    'D': 3,
    'L': -1,
    'R': 1
}

def manhattan_distance(state, goal_state):
    """Calculate Manhattan distance heuristic."""
    distance = 0
    for idx, tile in enumerate(state):
        if tile == 0:
            continue
        goal_idx = goal_state.index(tile)
        distance += abs(idx // 3 - goal_idx // 3) + abs(idx % 3 - goal_idx % 3)
    return distance

def get_neighbors(state):
    """Generate valid neighbor states with the move taken."""
    neighbors = []
    blank_index = state.index(0)
    row, col = divmod(blank_index, 3)

    for move, delta in MOVES.items():
        new_index = blank_index + delta

        if move == 'U' and row == 0:
            continue
        if move == 'D' and row == 2:
            continue
        if move == 'L' and col == 0:
            continue
        if move == 'R' and col == 2:
            continue

        new_state = list(state)
        new_state[blank_index], new_state[new_index] = new_state[new_index], new_state[blank_index]
        neighbors.append((tuple(new_state), move))
    return neighbors

def is_solvable(state, goal_state):
    """Check if the puzzle is solvable relative to the goal state."""
    inversion_count = 0
    tiles = [t for t in state if t != 0]
    goal_tiles = [t for t in goal_state if t != 0]
    goal_positions = {tile: idx for idx, tile in enumerate(goal_tiles)}

    mapped_tiles = [goal_positions[tile] for tile in tiles]
    for i in range(len(mapped_tiles)):
        for j in range(i + 1, len(mapped_tiles)):
            if mapped_tiles[i] > mapped_tiles[j]:
                inversion_count += 1
    return inversion_count % 2 == 0

def reconstruct_path(came_from, current):
    """Reconstruct path from start to goal."""
    moves = []
    states = []
    while current in came_from:
        current, move = came_from[current]
        moves.append(move)
        states.append(current)
    moves.reverse()
    states.reverse()
    return moves, states

def a_star(start_state, goal_state):
    """A* search for solving the 8 puzzle."""
    if not is_solvable(start_state, goal_state):
        return None, None

    frontier = []
    heapq.heappush(frontier, (manhattan_distance(start_state, goal_state), 0, start_state))
    came_from = {}
    g_score = {start_state: 0}
    visited = set()

    while frontier:
        _, g, current = heapq.heappop(frontier)

        if current == goal_state:
            moves, states = reconstruct_path(came_from, current)
            return moves, states

        visited.add(current)

        for neighbor, move in get_neighbors(current):
            if neighbor in visited:
                continue
            tentative_g = g + 1
            if tentative_g < g_score.get(neighbor, float('inf')):
                came_from[neighbor] = (current, move)
                g_score[neighbor] = tentative_g
                f_score = tentative_g + manhattan_distance(neighbor, goal_state)
                heapq.heappush(frontier, (f_score, tentative_g, neighbor))
    return None, None

def print_state(state):
    """Print puzzle state in 3x3 format."""
    for i in range(0, 9, 3):
        print(" ".join(str(x) if x != 0 else "_" for x in state[i:i+3]))
    print()

def get_state_from_user(prompt):
    """Get a valid puzzle state from user input."""
    while True:
        try:
            raw = input(prompt).strip().split()
            if len(raw) != 9:
                raise ValueError("You must enter exactly 9 numbers.")
            nums = list(map(int, raw))
            if sorted(nums) != list(range(9)):
                raise ValueError("Numbers must be from 0 to 8 with no duplicates.")
            return tuple(nums)
        except ValueError as e:
            print(f"Invalid input: {e}. Please try again.")

if __name__ == "__main__":
    print("Enter the initial state (0 for blank):")
    start_state = get_state_from_user("Initial state (e.g., '1 2 3 4 5 6 0 7 8'): ")

    print("Enter the goal state (0 for blank):")
    goal_state = get_state_from_user("Goal state (e.g., '1 2 3 4 5 6 7 8 0'): ")

    print("\nInitial State:")
    print_state(start_state)
    print("Goal State:")
    print_state(goal_state)

    moves, states = a_star(start_state, goal_state)

    if moves is None:
        print("This puzzle is unsolvable.")
    else:
        print(f"\nSolution found in {len(moves)} moves:")
        for i, (state, move) in enumerate(zip(states, moves), start=1):
            print(f"Move {i}: {move}")
            print_state(state)
        print("Goal State Reached!")

Enter the initial state (0 for blank):
Initial state (e.g., '1 2 3 4 5 6 0 7 8'): 2 8 3 1 6 3 7 0 5
Invalid input: Numbers must be from 0 to 8 with no duplicates.. Please try again.
Initial state (e.g., '1 2 3 4 5 6 0 7 8'): 2 8 3 1 6 4 7 0 5
Enter the goal state (0 for blank):
Goal state (e.g., '1 2 3 4 5 6 7 8 0'): 1 2 3 8 0 4 7 6 5

Initial State:
2 8 3
1 6 4
7 _ 5

Goal State:
1 2 3
8 _ 4
7 6 5


Solution found in 5 moves:
Move 1: U
2 8 3
1 6 4
7 _ 5

Move 2: U
2 8 3
1 _ 4
7 6 5

Move 3: L
2 _ 3
1 8 4
7 6 5

Move 4: D
_ 2 3
1 8 4
7 6 5

Move 5: R
1 2 3
_ 8 4
7 6 5

Goal State Reached!
